# Chapter 12 -- 汇总、保存与部署

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chaosfrey-arch/news-classification-tutorial/blob/main/chapter12_summary.ipynb)

**本章目标：** 对比四个模型的结果，保存最终模型，封装一个可直接调用的预测函数，并展望下一步。

**预计时间：** 30 分钟

## 12.1 四个模型的方法对比

在正式运行结果之前，先在概念层面回顾四个模型：

| | 朴素贝叶斯 | kNN | TextCNN | BiLSTM |
|---|---|---|---|---|
| **特征表示** | TF-IDF | TF-IDF + SVD | Embedding | Embedding |
| **学习机制** | 统计概率 | 距离+投票 | 局部卷积特征 | 序列循环记忆 |
| **训练速度** | 最快（秒级）| 较慢（需降维）| 快（GPU 友好）| 慢（顺序计算）|
| **可解释性** | 强（概率公式）| 强（找相似例子）| 弱 | 弱 |
| **优势** | 基准线，速度快 | 直觉清晰 | 短语模式检测 | 长距离上下文 |

**结论：** 对于新闻标题+摘要分类这类任务，TF-IDF + 朴素贝叶斯已经表现很好；深度学习的优势在更复杂的自然语言理解任务上更为明显。

In [ ]:
# 加载所有在前面章节里保存的模型和工具
import pandas as pd, numpy as np, re, joblib
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from tensorflow.keras.preprocessing.sequence import pad_sequences

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r'http\S+|www\S+|https\S+', '', t)
    t = re.sub(r'[^a-z\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

# 重建测试集（保证 random_state=42 与各章一致）
df = pd.read_csv('dataset.csv').dropna(subset=['Class']).copy()
df['Description'] = df['Description'].fillna('')
df['text'] = (df['Title'] + ' ' + df['Description']).apply(clean_text)
le = LabelEncoder()
y  = le.fit_transform(df['Class'])
_, X_test_text, _, y_test = train_test_split(
    df['text'].values, y, test_size=0.3, random_state=42, stratify=y)

# 加载 TF-IDF 模型（Chapter 6）
nb_model   = joblib.load('nb_model.pkl')
tfidf_vec  = joblib.load('tfidf_vectorizer.pkl')
X_test_tfidf = tfidf_vec.transform(X_test_text)

# 加载 kNN 相关资源（Chapter 7）
knn_model  = joblib.load('knn_model.pkl')
svd        = joblib.load('svd.pkl')
normalizer = joblib.load('normalizer.pkl')
X_test_knn = normalizer.transform(svd.transform(X_test_tfidf))

# 加载深度学习模型（Chapter 10, 11）
tokenizer  = joblib.load('tokenizer.pkl')
cnn_model  = tf.keras.models.load_model('cnn_model.keras')
bilstm_model = tf.keras.models.load_model('bilstm_model.keras')
X_test_pad = pad_sequences(
    tokenizer.texts_to_sequences(X_test_text),
    maxlen=100, padding='pre', truncating='pre')

print('所有模型加载完毕！')

In [ ]:
# 逐一评估，汇总结果
results = {}

# 朴素贝叶斯
y_nb  = nb_model.predict(X_test_tfidf)
results['Naive Bayes'] = {
    'accuracy': accuracy_score(y_test, y_nb),
    'macro_f1': f1_score(y_test, y_nb, average='macro')
}

# kNN
y_knn = knn_model.predict(X_test_knn)
results['kNN'] = {
    'accuracy': accuracy_score(y_test, y_knn),
    'macro_f1': f1_score(y_test, y_knn, average='macro')
}

# TextCNN
y_cnn = np.argmax(cnn_model.predict(X_test_pad, verbose=0), axis=1)
results['TextCNN'] = {
    'accuracy': accuracy_score(y_test, y_cnn),
    'macro_f1': f1_score(y_test, y_cnn, average='macro')
}

# BiLSTM
y_bilstm = np.argmax(bilstm_model.predict(X_test_pad, verbose=0), axis=1)
results['BiLSTM'] = {
    'accuracy': accuracy_score(y_test, y_bilstm),
    'macro_f1': f1_score(y_test, y_bilstm, average='macro')
}

print(f'{"模型":<15} {"Accuracy":>10} {"Macro F1":>10}')
print('-' * 38)
for name, m in results.items():
    print(f'{name:<15} {m["accuracy"]:>10.4f} {m["macro_f1"]:>10.4f}')

In [ ]:
import matplotlib.pyplot as plt

names = list(results.keys())
accs  = [results[n]['accuracy'] for n in names]
f1s   = [results[n]['macro_f1'] for n in names]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, accs, width, label='Accuracy',
               color=['steelblue','coral','mediumseagreen','mediumpurple'])
bars2 = ax.bar(x + width/2, f1s,  width, label='Macro F1',
               color=['steelblue','coral','mediumseagreen','mediumpurple'], alpha=0.6)

for bar in bars1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)

ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylim(0.75, 1.0)
ax.set_ylabel('Score')
ax.set_title('四个模型性能对比', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 12.2 封装预测函数

最终目标：给任意一篇新闻文章，能自动预测它属于哪个类别。

下面以朴素贝叶斯（效率最高、准确率最高的那个）为例封装一个完整的预测函数。

In [ ]:
# 封装一个完整的预测函数（以朴素贝叶斯为例，准确率最高且推理最快）
def predict_category(text, model=nb_model, vectorizer=tfidf_vec,
                     label_encoder=le):
    """
    输入：一篇新闻的标题或标题+摘要
    输出：预测类别名称 + 各类别概率
    """
    cleaned = clean_text(text)
    vec     = vectorizer.transform([cleaned])
    pred    = model.predict(vec)[0]
    proba   = model.predict_proba(vec)[0]
    return {
        'predicted_class': label_encoder.inverse_transform([pred])[0],
        'probabilities': dict(zip(label_encoder.classes_, proba.round(4)))
    }

# 测试几条样本
test_articles = [
    "Apple announces new iPhone with revolutionary AI chip and camera system",
    "Manchester United defeats Arsenal 3-1 in Premier League clash",
    "Federal Reserve raises interest rates amid inflation concerns",
    "NASA's Mars rover discovers signs of ancient water in rock samples",
]

for article in test_articles:
    result = predict_category(article)
    print(f'文章：{article[:60]}...')
    print(f'预测类别：{result["predicted_class"]}')
    probs = result['probabilities']
    for cls, p in sorted(probs.items(), key=lambda x: -x[1]):
        bar = '█' * int(p * 30)
        print(f'  {cls:15s}: {bar} {p:.4f}')
    print()

In [ ]:
# 也可以封装深度学习版本
def predict_category_deep(text, model=cnn_model, tok=tokenizer,
                          label_encoder=le, max_len=100):
    cleaned  = clean_text(text)
    seq      = tok.texts_to_sequences([cleaned])
    padded   = pad_sequences(seq, maxlen=max_len, padding='pre', truncating='pre')
    proba    = model.predict(padded, verbose=0)[0]
    pred_idx = np.argmax(proba)
    return {
        'predicted_class': label_encoder.inverse_transform([pred_idx])[0],
        'probabilities': dict(zip(label_encoder.classes_, proba.round(4)))
    }

# 测试
article = "Scientists discover a new species of dinosaur in Argentina"
result = predict_category_deep(article)
print(f'文章：{article}')
print(f'CNN 预测类别：{result["predicted_class"]}')
for cls, p in sorted(result['probabilities'].items(), key=lambda x: -x[1]):
    print(f'  {cls:15s}: {p:.4f}')

## 12.3 下一步可以做什么？

恭喜！你已经完成了这个教程的全部内容。以下是一些进阶方向：

### 提升模型性能
- **预训练词向量**：用 GloVe 或 FastText 初始化 Embedding 层，代替随机初始化
- **BERT / RoBERTa**：Transformer 架构的预训练模型，目前最强的文本分类方法，准确率可达 95%+
- **数据增强**：对少数类别做同义词替换、回译等增强

### 工程化部署
- **FastAPI**：把预测函数包装成 REST API，让其他程序调用
- **Streamlit**：做一个带界面的 Web Demo，直接粘贴文章就能看预测结果
- **Docker**：容器化部署，随时随地运行

### 延伸学习
- [Hugging Face 课程](https://huggingface.co/learn/nlp-course)：学习 Transformer 和 BERT
- [fast.ai](https://www.fast.ai/)：面向实践的深度学习课程
- [Kaggle NLP 竞赛](https://www.kaggle.com/competitions?search=nlp)：用真实比赛练手

## 全教程总结

你学完了一条完整的 NLP 实验流水线：

```
原始数据（dataset.csv）
    ↓
Chapter 1-2：用 pandas 探索数据，用 matplotlib 可视化
    ↓
Chapter 3：文本预处理（正则清洗、划分训练/测试集）
    ↓
Chapter 4：理解模型、训练、预测的概念
    ↓
Chapter 5：TF-IDF 把文字变成数字
    ↓
Chapter 6：朴素贝叶斯（Accuracy ≈ 90%）
Chapter 7：kNN + SVD 降维（Accuracy ≈ 84%）
    ↓
Chapter 8：神经网络基础（反向传播、Keras）
Chapter 9：词向量 Embedding（Tokenizer、pad_sequences）
    ↓
Chapter 10：TextCNN（Accuracy ≈ 91%）
Chapter 11：BiLSTM（Accuracy ≈ 91-92%）
    ↓
Chapter 12：汇总对比、封装预测函数、部署展望
```

**恭喜完成全部教程！** 你已经掌握了从零到深度学习文本分类的完整技能栈。